# Clasificación: Predicción de listados de precios altos

**Rol de canalización:** Cuaderno obligatorio 05, el paso de modelado de clasificación top-price.

Este cuaderno entrena clasificadores para la señal externa `top_price`, guarda informes y escribe predicciones en BigQuery. Los resultados y la configuración del modelo se conservan para la línea base de enseñanza.

**Consume:** BigQuery `vw_classification_dataset`, donde `top_price` se deriva del mercado observado `price_label`.
**Produce:** informes CSV/PNG, `fact_top_price_predictions` y `model_evaluation_summary`.
**Feeds:** `vw_bi_dashboard` y la capa de soporte de decisiones Streamlit.

La clasificación estima la probabilidad top-price. Permanece separado de la regresión y no se deriva de los residuos expected-price.

### Descripción general funcional
Entradas: datos de listado con etiquetas de precios de BigQuery.
Proceso: preprocesar características, entrenar clasificadores, evaluarlos y guardar artefactos.
Salidas: informes CSV/PNG y tablas BigQuery para predicciones y evaluación.
Motivo: la clasificación enseña cómo predecir una categoría (top-price) en lugar de un número continuo.

**Objetivo:** Entrenar clasificadores para predecir listados top-price.
**Entradas:** BigQuery listados con etiquetas de precios.
**Proceso:** Cree un conjunto de datos, preprocese funciones, entrene modelos, evalúe y guarde artefactos.
**Salidas:** Informes CSV/PNG y tablas BigQuery para predicciones y evaluación.
**Por qué:** La clasificación introduce umbrales de decisión y predicción categórica.


## 1. Importaciones y configuración

La configuración está centralizada para mantener las rutas y los ID consistentes a lo largo del curso.

### Configuración
Importamos todas las herramientas, configuramos la semilla aleatoria y definimos directorios de salida. Mantener estos valores centralizados hace que el cuaderno sea más fácil de seguir y actualizar.

### Por qué estos modelos
La regresión logística es una línea de base clara, el bosque aleatorio captura las no linealidades y el aumento de gradiente ofrece un rendimiento sólido con un ajuste mínimo.

**Objetivo:** Importar herramientas y definir constantes de configuración.
**Entradas:** Bibliotecas, semilla aleatoria, identificadores de proyecto.
**Proceso:** Cargar módulos y crear una carpeta de salida de informes.
**Resultados:** Entorno listo para modelado y generación de informes.
**Por qué:** La configuración central hace que el cuaderno sea más fácil de seguir.


In [ ]:
# Objetivo: importar bibliotecas, establecer configuraciones y preparar carpetas de salida.  # Objetivo
# Entrada: bibliotecas principales, herramientas scikit-learn, cliente BigQuery y constantes.  # Aporte
# Proceso: importar módulos, definir constantes y crear el directorio de informes.  # Proceso
# Salida: Entorno inicializado para clasificación y generación de informes.  # Producción

from pathlib import Path  # Path handles file paths safely.
from datetime import datetime, timezone  # datetime provides timestamps and timezone utilities.

import numpy as np  # numpy supports numeric operations and reproducibility.
import pandas as pd  # pandas handles tabular data.
import matplotlib.pyplot as plt  # matplotlib creates plots.

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate  # Split and CV tools.
from sklearn.compose import ColumnTransformer  # Column-wise preprocessing.
from sklearn.pipeline import Pipeline  # Pipeline para encadenar preprocesamiento y modelos.
from sklearn.impute import SimpleImputer  # Imputacion de valores faltantes.
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Encoding and scaling.
from sklearn.linear_model import LogisticRegression  # Clasificador de regresion logistica.
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier  # Tree-based models.
from sklearn.metrics import (  # Metricas de evaluacion para clasificacion.
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from google.cloud import bigquery  # BigQuery client for data access and writes.



### Configuración y entorno
Mantenemos todas las constantes y la configuración del cliente BigQuery centralizadas.


In [ ]:
# Configuración cargada desde project_config.yaml compartido


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


REPO_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(REPO_ROOT / 'config' / 'project_config.yaml')

RANDOM_SEED = int(PROJECT_CONFIG.get('random_state', 42))
project_id = str(PROJECT_CONFIG.get('gcp_project_id', 'albertheadofdata101')).strip()
dataset_id = str(PROJECT_CONFIG.get('bq_dataset', 'autoscout_audi_a3_germany')).strip()
make = str(PROJECT_CONFIG.get('make', 'audi')).strip().lower()
model = str(PROJECT_CONFIG.get('model', 'a3')).strip().lower()
country_name = str(PROJECT_CONFIG.get('country', 'germany')).strip().lower()
TAG = f'{make}_{model}_{country_name}'

REPORTS_DIR = REPO_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Cargue el conjunto de datos de etiquetas top_price externo desde BigQuery


In [ ]:
# Objetivo: cargar el conjunto de datos top_price etiquetado externamente desde BigQuery.  # Objetivo
# Entrada: project_id, dataset_id y alcance de marca/modelo/país configurado.  # Aporte
# Proceso: Consulta vw_classification_dataset y conserva la etiqueta top_price de price_label.  # Proceso
# Salida: df que contiene características y el objetivo binario top_price preexistente.  # Producción

client = bigquery.Client(project=project_id)


def run_query(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()


sql_dataset = f'''
SELECT
  listing_id,
  make,
  model,
  listing_country,
  actual_price_eur,
  mileage_km,
  age_years,
  power_hp,
  fuel_type,
  price_label,
  top_price
FROM `{project_id}.{dataset_id}.vw_classification_dataset`
WHERE LOWER(make) = '{make}'
  AND LOWER(model) = '{model}'
  AND LOWER(listing_country) = '{country_name}'
'''

df = run_query(sql_dataset)

if len(df) == 0:
    sql_dataset_no_country = f'''
    SELECT
      listing_id,
      make,
      model,
      listing_country,
      actual_price_eur,
      mileage_km,
      age_years,
      power_hp,
      fuel_type,
      price_label,
      top_price
    FROM `{project_id}.{dataset_id}.vw_classification_dataset`
    WHERE LOWER(make) = '{make}'
      AND LOWER(model) = '{model}'
    '''
    df = run_query(sql_dataset_no_country)
    print('No rows found for configured country; fallback scope used make/model only.')

if len(df) == 0:
    raise ValueError('No rows available for configured classification scope. Check data and config values.')

print('Loaded rows:', len(df))
print('Scope:', make, model, country_name)


## 3. Selección de funciones y filtros de validez básicos.

La lista de funciones está fijada para evitar fugas de objetivos y mantener los resultados estables.

### Selección de funciones
Utilizamos una lista de funciones fija para evitar fugas y mantener la coherencia de los resultados. También filtramos valores no válidos.

### Justificación de la característica
Las características elegidas son numéricas y están disponibles para todos los listados, lo que las convierte en entradas confiables para un clasificador principiante.

**Objetivo:** Seleccionar características del modelo y filtrar filas no válidas.
**Entradas:** Columnas del marco de datos para precio, kilometraje, antigüedad, combustible y potencia.
**Proceso:** Validar columnas y eliminar valores imposibles.
**Salidas:** Limpiar X e y listos para el preprocesamiento.
**Por qué:** La calidad de la entrada afecta directamente el rendimiento y la estabilidad del modelo.


In [ ]:
# Objetivo: Seleccionar características y aplicar filtros de validez básicos.  # Objetivo
# Entrada: df con características de listado y objetivo externo top_price.  # Aporte
# Proceso: definir la lista de funciones, validar las columnas requeridas y filtrar valores numéricos no válidos.  # Proceso
# Salida: X (características) e y (objetivo) filtrados para modelado.  # Producción

feature_cols = [
    'actual_price_eur',
    'mileage_km',
    'age_years',
    'power_hp',
    'fuel_type',
    'listing_country',
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required feature columns: {missing}')

y = df['top_price'].astype(int)
X = df[feature_cols].copy()

mask = pd.Series([True] * len(X))
if 'actual_price_eur' in X.columns:
    mask &= X['actual_price_eur'] > 0
if 'mileage_km' in X.columns:
    mask &= X['mileage_km'] > 0
if 'age_years' in X.columns:
    mask &= X['age_years'] >= 0
if 'power_hp' in X.columns:
    mask &= X['power_hp'] > 0

X = X.loc[mask].copy()
y = y.loc[mask].copy()


## 4. Tubería de preprocesamiento

El preprocesamiento es explícito para que los estudiantes vean cómo se manejan los datos numéricos y categóricos.

### Preprocesamiento
Las columnas numéricas se imputan y se escalan, y las columnas categóricas se imputan y se codifican en caliente. Esto crea una matriz numérica limpia para modelar.

### Justificación del preprocesamiento
Imputamos valores faltantes y escalamos características numéricas para que los modelos se comporten de manera predecible. La codificación one-hot hace que los valores categóricos sean utilizables por los algoritmos ML.

**Objetivo:** Preparar el preprocesamiento de datos numéricos y categóricos.
**Entradas:** Matriz de características con tipos mixtos.
**Proceso:** Imputa valores faltantes, escala características numéricas, categorías de codificación one-hot.
**Salidas:** Un transformador de preprocesamiento reutilizable.
**Por qué:** Los modelos requieren entradas numéricas limpias y codificación consistente.


In [ ]:
# Objetivo: crear canales de preprocesamiento para características numéricas y categóricas.  # Objetivo
# Entrada: matriz de características X con tipos de datos mixtos.  # Aporte
# Proceso: defina canalizaciones numéricas y categóricas y combínelas con ColumnTransformer.  # Proceso
# Salida: preprocesador que se puede utilizar en tuberías modelo.  # Producción

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()  # Identify numeric columns.
cat_cols = [c for c in X.columns if c not in numeric_cols]  # Identify categorical columns.

numeric_transformer = Pipeline(steps=[  # Pipeline para variables numericas.
    ('imputer', SimpleImputer(strategy='median')),  # Fill missing numeric values.
    ('scaler', StandardScaler()),  # Standardize numeric values.
])

categorical_transformer = Pipeline(steps=[  # Pipeline para variables categoricas.
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Fill missing categories.
    ('onehot', OneHotEncoder(handle_unknown='ignore')),  # One-hot encode categories.
])

preprocessor = ColumnTransformer(  # Combine numeric and categorical pipelines.
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, cat_cols),
    ],
    remainder='drop',  # Drop any columns not specified.
)


## 5. División tren-prueba

Una división estratificada preserva el equilibrio de clases y hace que las métricas sean más confiables.

### División de prueba de tren
Una división estratificada mantiene el equilibrio de clases similar entre el tren y los conjuntos de prueba.

### Enfoque de evaluación
Usamos una división estratificada para preservar el equilibrio de clases, luego evaluamos con exactitud, precisión, recuperación, F1 y ROC-AUC.

**Objetivo:** Dividir el conjunto de datos para su evaluación.
**Entradas:** X e y.
**Proceso:** Tren estratificado/división de prueba con una semilla fija.
**Salidas:** Conjuntos de entrenamiento y prueba.
**Por qué:** Un conjunto de pruebas retenido proporciona una evaluación imparcial.


In [ ]:
# Objetivo: dividir el conjunto de datos en conjuntos de entrenamiento y de prueba.  # Objetivo
# Entrada: matriz de características X y vector objetivo y.  # Aporte
# Proceso: realice una división estratificada de tren/prueba con una semilla aleatoria fija.  # Proceso
# Salida: X_train, X_test, y_train, y_test para modelado y evaluación.  # Producción

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)  # Split data with class balance preserved.


## 6. Definiciones de modelos

Comparamos tres clasificadores estándar con hiperparámetros consistentes.

### Modelos
Definimos tres modelos: regresión logística, bosque aleatorio y aumento de gradiente. Esto brinda a los estudiantes opciones tanto lineales como no lineales.

**Objetivo:** Definir tres modelos de clasificación.
**Entradas:** Transformador de preprocesamiento e información de equilibrio de clases.
**Proceso:** Configurar regresión logística, bosque aleatorio y canalizaciones de aumento de gradiente.
**Resultados:** Un diccionario de canalizaciones modelo.
**Por qué:** Varios modelos permiten comparar enfoques lineales y no lineales.


In [ ]:
# Objetivo: Definir los modelos de clasificación y sus configuraciones.  # Objetivo
# Entrada: Etiquetas de entrenamiento para estimar el desequilibrio de clases.  # Aporte
# Proceso: calcular la proporción de clases, establecer hiperparámetros, crear canalizaciones.  # Proceso
# Salida: diccionario de modelos que contiene tres canalizaciones de modelos.  # Producción

train_counts = y_train.value_counts()  # Count class frequencies in training data.
minority_ratio = train_counts.min() / train_counts.sum()  # Compute minority class ratio.
use_class_weight = minority_ratio < 0.30  # Decide whether to apply class weights.

logreg_params = {  # Parametros para regresion logistica.
    'max_iter': 2000,
    'solver': 'lbfgs',
}
if use_class_weight:  # If class imbalance is large,
    logreg_params['class_weight'] = 'balanced'  # Usa pesos de clase balanceados.

model_a = Pipeline(steps=[  # Pipeline para regresion logistica.
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(**logreg_params)),
])

rf_params = {  # Parametros para random forest.
    'n_estimators': 300,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
}
if use_class_weight:  # If class imbalance is large,
    rf_params['class_weight'] = 'balanced_subsample'  # Usa pesos balanceados por submuestra.

model_b = Pipeline(steps=[  # Pipeline para random forest.
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(**rf_params)),
])

model_c = Pipeline(steps=[  # Pipeline para gradient boosting.
    ('preprocess', preprocessor),
    ('clf', HistGradientBoostingClassifier(random_state=RANDOM_SEED)),
])

models = {  # Dictionary of model name to pipeline.
    'LogisticRegression': model_a,
    'RandomForest': model_b,
    'HistGradientBoosting': model_c,
}


### Control de cordura
Antes del entrenamiento, confirmamos el desequilibrio de clases y la dificultad base.


## 7. Validación cruzada (solo capacitación)

La validación cruzada se ejecuta solo con datos de entrenamiento para evitar fugas de pruebas.

### Validación cruzada
Realizamos validación cruzada solo en el conjunto de entrenamiento para estimar la estabilidad del modelo sin filtrar datos de prueba.

### Propósito de validación cruzada
CV proporciona una verificación de estabilidad al evaluar modelos en múltiples pliegues de entrenamiento.

**Objetivo:** Estimar la estabilidad del modelo con validación cruzada.
**Entradas:** Datos de capacitación y canalizaciones de modelos.
**Proceso:** Ejecutar CV estratificado y resumir ROC-AUC y F1.
**Salidas:** Tabla resumen de CV.
**Por qué:** El CV muestra cómo varían los resultados entre los pliegues.


In [ ]:
# Objetivo: ejecutar una validación cruzada de los datos de entrenamiento si es posible.  # Objetivo
# Entrada: X_train, y_train y diccionario de modelos.  # Aporte
# Proceso: elija pliegues, ejecute CV, calcule resúmenes ROC-AUC y F1.  # Proceso
# Salida: tabla de comparación_cv con métricas medias y estándar.  # Producción

train_counts = y_train.value_counts()  # Count class frequencies in training data.
minority_count = int(train_counts.min()) if len(train_counts) > 0 else 0  # Determine minority count.

if y_train.nunique() < 2 or minority_count < 2:  # If not enough class variety,
    comparison_cv = pd.DataFrame({  # Crea una tabla resumen provisional.
        'model': list(models.keys()),
        'cv_roc_auc_mean': np.nan,
        'cv_roc_auc_std': np.nan,
        'cv_f1_mean': np.nan,
        'cv_f1_std': np.nan,
    })
else:  # If we have enough data for CV,
    n_splits = min(5, minority_count)  # Usa hasta 5 folds.
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)  # Stratified CV.

    cv_rows = []  # Collect CV results per model.
    for name, pipeline in models.items():  # Loop through each model pipeline.
        scores = cross_validate(  # Ejecuta validacion cruzada.
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring={'roc_auc': 'roc_auc', 'f1': 'f1'},
            n_jobs=-1,
        )
        cv_rows.append({  # Guarda estadisticos resumen de este modelo.
            'model': name,
            'cv_roc_auc_mean': scores['test_roc_auc'].mean(),
            'cv_roc_auc_std': scores['test_roc_auc'].std(),
            'cv_f1_mean': scores['test_f1'].mean(),
            'cv_f1_std': scores['test_f1'].std(),
        })

    comparison_cv = pd.DataFrame(cv_rows)  # Convierte las filas de CV en un DataFrame.

print(comparison_cv)  # Display CV summary.


## 8. Evaluación de pruebas y tabla comparativa.

La tabla de comparación se guarda para informes y discusión posterior.

### Evaluación de prueba
Ajustamos cada modelo y calculamos exactitud, precisión, recuperación, F1 y ROC-AUC en el conjunto de prueba. La tabla combinada se guarda.

**Objetivo:** Evaluar modelos en el conjunto de prueba y guardar la tabla de comparación.
**Entradas:** Entrenar/probar divisiones y modelar tuberías.
**Proceso:** Ajustar, predecir, calcular métricas y guardar CSV.
**Salidas:** `model_comparison.csv` en informes.
**Por qué:** Una tabla de comparación guardada admite informes y revisiones.


In [ ]:
# Objetivo: evaluar cada modelo en el conjunto de prueba y guardar la tabla de comparación.  # Objetivo
# Entrada: X_train, y_train, X_test, y_test y diccionario de modelos.  # Aporte
# Proceso: ajustar modelos, predecir, calcular métricas, fusionar con resumen de CV, guardar CSV.  # Proceso
# Salida: tabla de comparación guardada en informes e impresa.  # Producción

# Ajustar y evaluar en el conjunto de prueba # Usar datos de prueba para la evaluación final.

test_rows = []  # Collect test metrics per model.
for name, pipeline in models.items():  # Loop through models.
    pipeline.fit(X_train, y_train)  # Fit model on training data.
    y_proba = pipeline.predict_proba(X_test)[:, 1]  # Predict probabilities on test data.
    y_pred = (y_proba >= 0.5).astype(int)  # Convierte probabilidades en etiquetas de clase.

    test_rows.append({  # Guarda metricas para este modelo.
        'model': name,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred, zero_division=0),
        'test_recall': recall_score(y_test, y_pred, zero_division=0),
        'test_f1': f1_score(y_test, y_pred, zero_division=0),
        'test_roc_auc': roc_auc_score(y_test, y_proba),
    })

comparison_test = pd.DataFrame(test_rows)  # Convierte metricas de test en DataFrame.
comparison = comparison_test.merge(comparison_cv, on='model', how='left')  # Merge with CV summary.

comparison_path = REPORTS_DIR / f"{TAG}_model_comparison.csv"  # Define la ruta del archivo de salida.
comparison.to_csv(comparison_path, index=False)  # Guarda la tabla comparativa en CSV.
print('Saved comparison table to:', comparison_path)  # Confirm save location.


## 9. Matrices de confusión (guardadas como PNG)

Las matrices de confusión se guardan como PNG para diapositivas o informes del curso.

### Matrices de confusión
Las matrices de confusión muestran clases verdaderas versus clases predichas y se guardan como PNG.

### Propósito de la matriz de confusión
Esto muestra dónde el modelo hace predicciones correctas e incorrectas, lo cual es fácil de interpretar para los principiantes.

**Objetivo:** Crear matrices de confusión para cada modelo.
**Entradas:** Etiquetas de prueba y predicciones.
**Proceso:** Calcule matrices de confusión, trace y guarde archivos PNG.
**Resultados:** Una imagen de matriz de confusión por modelo.
**Por qué:** Las matrices de confusión proporcionan desgloses de errores intuitivos.


In [ ]:
# Objetivo: Trazar y guardar matrices de confusión para cada modelo.  # Objetivo
# Entrada: X_test, y_test y canalizaciones entrenadas.  # Aporte
# Proceso: predecir etiquetas, calcular matrices de confusión, trazar y guardar archivos PNG.  # Proceso
# Salida: Imágenes de matriz de confusión guardadas en la carpeta de informes.  # Producción

confusion_paths = []  # Collect saved confusion matrix paths.

for name, pipeline in models.items():  # Loop through models.
    y_proba = pipeline.predict_proba(X_test)[:, 1]  # Predict probabilities on test set.
    y_pred = (y_proba >= 0.5).astype(int)  # Convierte a etiquetas predichas.

    cm = confusion_matrix(y_test, y_pred)  # Compute confusion matrix.

    plt.figure(figsize=(4, 4))  # Crea una figura nueva.
    plt.imshow(cm, cmap='Blues')  # Display matrix as an image.
    plt.title(f'Confusion Matrix ({TAG}) - {name}')  # Title with model name.
    plt.xlabel('Predicted')  # Label x-axis.
    plt.ylabel('Actual')  # Label y-axis.

    for i in range(cm.shape[0]):  # Loop through rows.
        for j in range(cm.shape[1]):  # Loop through columns.
            plt.text(j, i, cm[i, j], ha='center', va='center', color='black')  # Add counts.

    plt.tight_layout()  # Adjust layout.

    file_safe_name = name.replace(' ', '_')  # Crea un nombre de archivo seguro.
    cm_path = REPORTS_DIR / f"{TAG}_confusion_matrix_{file_safe_name}.png"  # Output path.
    plt.savefig(cm_path, dpi=150)  # Guarda el grafico.
    plt.show()  # Display the plot.

    confusion_paths.append(cm_path)  # Guarda la ruta generada.


## 10. Análisis de umbral para el mejor modelo.

El análisis de umbral muestra cómo se compensan la precisión y la recuperación para obtener el mejor modelo.

### Análisis de umbral
Evaluamos cómo cambian la precisión y la recuperación a medida que movemos el umbral de decisión para el mejor modelo.

### Propósito del análisis de umbral
Cambiar el umbral de probabilidad revela el equilibrio entre precisión y recuperación, que es crucial en los problemas de clasificación.

**Objetivo:** Analizar las compensaciones entre precisión y recuperación entre umbrales.
**Entradas:** Mejores probabilidades del modelo en el conjunto de prueba.
**Proceso:** Calcule métricas en múltiples umbrales y guarde gráficos.
**Salidas:** Umbral CSV y gráficos de precisión/recuperación.
**Por qué:** El ajuste del umbral es clave para las decisiones de clasificación.


### El umbral es una decisión operativa
Variamos el umbral de probabilidad para intercambiar precisión versus recuperación sin reentrenamiento.


In [ ]:
# Objetivo: Analizar la precisión/recuperación a través de umbrales para obtener el mejor modelo.  # Objetivo
# Entrada: tabla de comparación y predicciones del modelo sobre datos de prueba.  # Aporte
# Proceso: seleccione el mejor modelo, calcule métricas para múltiples umbrales, guarde tablas y gráficos.  # Proceso
# Salida: tabla de umbrales CSV y gráficos de precisión/recuperación.  # Producción

# Mejor modelo por ROC-AUC, desempate por F1 # Seleccione el modelo con mejor rendimiento.
best_row = comparison.sort_values(
    ['test_roc_auc', 'test_f1'], ascending=False
).iloc[0]  # Take top row after sorting.

best_model_name = best_row['model']  # Name of the best model.
best_model = models[best_model_name]  # Best model pipeline.

best_proba = best_model.predict_proba(X_test)[:, 1]  # Probabilities for best model.


In [ ]:
thresholds = np.arange(0.2, 0.81, 0.05)  # Thresholds to evaluate.

rows = []  # Collect threshold metrics.
for t in thresholds:  # Loop over each threshold.
    preds = (best_proba >= t).astype(int)  # Convierte probabilidades en etiquetas.
    rows.append({  # Guarda metricas para este umbral.
        'threshold': t,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'accuracy': accuracy_score(y_test, preds),
    })

threshold_df = pd.DataFrame(rows)  # Convierte metricas en DataFrame.



In [ ]:
threshold_path = REPORTS_DIR / f"{TAG}_threshold_table_{best_model_name.replace(' ', '_')}.csv"  # Output path.
threshold_df.to_csv(threshold_path, index=False)  # Guarda la tabla de umbrales.
print('Saved threshold table to:', threshold_path)  # Confirm save.

plt.figure(figsize=(6, 4))  # Crea la figura para el grafico de precision.
plt.plot(threshold_df['threshold'], threshold_df['precision'], marker='o')  # Dibuja la precision.
plt.title(f'Precision vs Threshold ({TAG}) - {best_model_name}')  # Title.
plt.xlabel('Threshold')  # X-axis label.
plt.ylabel('Precision')  # Y-axis label.
plt.grid(True, alpha=0.3)  # Light grid for readability.
plt.tight_layout()  # Adjust layout.

precision_plot_path = REPORTS_DIR / f"{TAG}_precision_threshold_{best_model_name.replace(' ', '_')}.png"  # Output path.
plt.savefig(precision_plot_path, dpi=150)  # Guarda el grafico de precision.
plt.show()  # Display precision plot.

plt.figure(figsize=(6, 4))  # Crea la figura para el grafico de recall.
plt.plot(threshold_df['threshold'], threshold_df['recall'], marker='o')  # Dibuja el recall.
plt.title(f'Recall vs Threshold ({TAG}) - {best_model_name}')  # Title.
plt.xlabel('Threshold')  # X-axis label.
plt.ylabel('Recall')  # Y-axis label.
plt.grid(True, alpha=0.3)  # Light grid for readability.
plt.tight_layout()  # Adjust layout.

recall_plot_path = REPORTS_DIR / f"{TAG}_recall_threshold_{best_model_name.replace(' ', '_')}.png"  # Output path.
plt.savefig(recall_plot_path, dpi=150)  # Guarda el grafico de recall.
plt.show()  # Display recall plot.


## 11. Característica de artefactos importantes

Los resultados de interpretabilidad se guardan para su explicación en el aula.

### Interpretación de características
Guardamos los coeficientes de regresión logística y las importancias forestales aleatorias para facilitar la interpretación.

**Objetivo:** Guardar artefactos de interpretación de características.
**Insumos:** Regresión logística y modelos forestales aleatorios.
**Proceso:** Extraer coeficientes e importancias, guardar en CSV.
**Salidas:** Dos archivos CSV para facilitar la interpretación.
**Por qué:** La interpretación ayuda a los estudiantes a comprender el comportamiento modelo.


In [ ]:
# Objetivo: Guardar artefactos de importancia de características para su interpretación.  # Objetivo
# Entrada: Regresión logística entrenada y modelos forestales aleatorios.  # Aporte
# Proceso: extraiga los nombres de las características, los coeficientes y las importancias, guárdelos como CSV.  # Proceso
# Salida: Coeficiente e importancia de archivos CSV en informes.  # Producción

# Coeficientes de regresión logística # Interpretar modelo lineal.
logreg_pipe = models['LogisticRegression']  # Accede al pipeline de regresion logistica.
preprocessor_fitted = logreg_pipe.named_steps['preprocess']  # Accede al preprocesador ajustado.
clf = logreg_pipe.named_steps['clf']  # Accede al clasificador de regresion logistica.

feature_names = preprocessor_fitted.get_feature_names_out()  # Get feature names after encoding.
coefs = clf.coef_.ravel()  # Get coefficient values as a flat array.

coef_df = pd.DataFrame({  # Construye la tabla de coeficientes.
    'feature': feature_names,
    'coefficient': coefs,
}).sort_values('coefficient', ascending=False)  # Sort by coefficient.

coef_path = REPORTS_DIR / f"{TAG}_logreg_top_coefficients.csv"  # Output path for coefficients.
coef_df.to_csv(coef_path, index=False)  # Guarda el CSV de coeficientes.
print('Saved coefficients to:', coef_path)  # Confirm save.

# Importancias de las características del bosque aleatorio # Interpretar el modelo basado en árboles.
rf_pipe = models['RandomForest']  # Accede al pipeline de random forest.
rf_clf = rf_pipe.named_steps['clf']  # Accede al clasificador random forest.

rf_feature_names = rf_pipe.named_steps['preprocess'].get_feature_names_out()  # Feature names after encoding.
rf_importances = rf_clf.feature_importances_  # Importance scores from random forest.

rf_imp_df = pd.DataFrame({  # Construye la tabla de importancias.
    'feature': rf_feature_names,
    'importance': rf_importances,
}).sort_values('importance', ascending=False)  # Sort by importance.

rf_imp_path = REPORTS_DIR / f"{TAG}_rf_feature_importance.csv"  # Output path for importances.
rf_imp_df.to_csv(rf_imp_path, index=False)  # Guarda el CSV de importancias.
print('Saved RF importances to:', rf_imp_path)  # Confirm save.


### Contrato de persistencia (el esquema debe permanecer estable)
Este paso escribe predicciones por listado con un esquema fijo utilizado por BI.


## 12. Persistir en las predicciones para BigQuery

Las predicciones se escriben en BigQuery con el esquema requerido.

### BigQuery persistencia
Escribimos probabilidades de predicción y resúmenes de evaluación en BigQuery con nombres de tablas y esquemas fijos.

### Propósito de persistencia
Guardar predicciones y resúmenes de evaluación garantiza que los resultados se puedan utilizar en paneles o análisis adicionales.

**Objetivo:** Escriba predicciones por listado en BigQuery.
**Entradas:** Conjunto de datos completo y modelos entrenados.
**Proceso:** Predecir probabilidades, ensamblar esquema, sobrescribir tabla.
**Salidas:** BigQuery tabla `fact_top_price_predictions`.
**Por qué:** Las predicciones persistentes admiten análisis posteriores.


In [ ]:
# Objetivo: conservar las predicciones por listado hasta BigQuery.  # Objetivo
# Entrada: conjunto completo de funciones X y canalizaciones de modelos entrenados.  # Aporte
# Proceso: predecir probabilidades para cada modelo, crear una tabla combinada, escribir en BigQuery.  # Proceso
# Salida: tabla fact_top_price_predictions en BigQuery.  # Producción

model_name_map = {  # Map pipeline names to schema values.
    'LogisticRegression': 'logistic_regression',
    'RandomForest': 'random_forest',
    'HistGradientBoosting': 'hist_gradient_boosting',
}

# Alinee listing_id con el conjunto de funciones filtradas # Mantenga los ID consistentes con X.
# Usamos df filtrado por X.index para mantener la coherencia con el preprocesamiento.  # Preservar la alineación.
df_modeling = df.loc[X.index].copy()  # Subset df to the same rows as X.
if 'listing_id' not in df_modeling.columns:  # Valida la presencia de listing_id.
    raise ValueError('listing_id is required for BigQuery persistence.')  # Stop if missing.

threshold_used = 0.5  # Fixed threshold used for predicted_label.

pred_rows = []  # Collect per-model prediction tables.
for name, pipeline in models.items():  # Loop through models.
    proba = pipeline.predict_proba(X)[:, 1]  # Predict probabilities for all rows.
    model_df = pd.DataFrame({  # Construye filas de prediccion para este modelo.
        'listing_id': df_modeling['listing_id'].astype('int64'),
        'model_name': model_name_map[name],
        'predicted_proba': proba.astype('float64'),
        'predicted_label': (proba >= threshold_used),
        'threshold_used': float(threshold_used),
    })
    pred_rows.append(model_df)  # Append to list.

predictions_df = pd.concat(pred_rows, ignore_index=True)  # Combine all model predictions.

pred_table = f"{project_id}.{dataset_id}.fact_top_price_predictions"  # Target table name.
client.delete_table(pred_table, not_found_ok=True)  # Delete existing table if present.
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')  # Overwrite on write.

client.load_table_from_dataframe(
    predictions_df,
    pred_table,
    job_config=job_config,
).result()  # Write predictions to BigQuery.

print('Rows written:', len(predictions_df))  # Confirm row count.
print('Table replaced:', pred_table)  # Confirm table name.


### Artefacto de gobernanza del modelo
Mantenemos las métricas de evaluación para que BI pueda comparar modelos a lo largo del tiempo.


## 13. Persistir resumen de evaluación a BigQuery

La tabla de resumen de la evaluación alimenta los paneles y el análisis posterior.

**Objetivo:** Escribir resumen de evaluación en BigQuery.
**Entradas:** Tabla de métricas de comparación.
**Proceso:** Cambiar el nombre de los campos, ordenar las columnas, sobrescribir la tabla de resumen.
**Salidas:** BigQuery tabla `model_evaluation_summary`.
**Por qué:** Las métricas de evaluación centralizadas son útiles para generar informes.


In [ ]:
# Objetivo: conservar las métricas de resumen de evaluación en BigQuery.  # Objetivo
# Entrada: tabla comparativa con métricas de prueba y CV.  # Aporte
# Proceso: cambiar el nombre de las columnas, agregar umbral, ordenar el esquema, escribir en BigQuery.  # Proceso
# Salida: tabla model_evaluación_summary en BigQuery.  # Producción

summary_df = comparison.copy()  # Copy the comparison table.
summary_df = summary_df.rename(columns={  # Rename columns to match schema.
    'model': 'model_name',
    'test_roc_auc': 'roc_auc_test',
    'test_f1': 'f1_test',
    'test_precision': 'precision_test',
    'test_recall': 'recall_test',
    'test_accuracy': 'accuracy_test',
})

summary_df['model_name'] = summary_df['model_name'].map(model_name_map)  # Map model names.
summary_df['threshold_used'] = float(threshold_used)  # Add threshold used.

summary_df = summary_df[[  # Order columns to match expected schema.
    'model_name',
    'roc_auc_test',
    'f1_test',
    'precision_test',
    'recall_test',
    'accuracy_test',
    'cv_roc_auc_mean',
    'cv_roc_auc_std',
    'cv_f1_mean',
    'cv_f1_std',
    'threshold_used',
]]

eval_table = f"{project_id}.{dataset_id}.model_evaluation_summary"  # Target summary table.
client.delete_table(eval_table, not_found_ok=True)  # Delete existing table if present.
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')  # Overwrite on write.

client.load_table_from_dataframe(
    summary_df,
    eval_table,
    job_config=job_config,
).result()  # Write summary to BigQuery.

print('Rows written:', len(summary_df))  # Confirm row count.
print('Table replaced:', eval_table)  # Confirm table name.
